In [15]:
import xarray as xr
import matplotlib.pyplot as plt
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import numpy as np
import cartopy.crs as ccrs

In [17]:
cluster = PBSCluster(
    cores=1,
    memory='32GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='2:00:00'
)
cluster.scale(jobs=5)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.67:43339,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# import data

In [18]:
Q = xr.open_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/Q')
Q

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 753888, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
    level      float64 8B ...
Data variables:
    Q          (time, latitude, longitude) float32 30GB dask.array<chunksize=(62824, 82, 121), meta=np.ndarray>

In [3]:
U = xr.open_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/U10')
V = xr.open_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/V10')

In [19]:
years = np.arange(1940, 2026, 1)
U = U.sel(time=U.time.dt.year.isin(years))
V = V.sel(time=V.time.dt.year.isin(years))

In [20]:
U

<xarray.Dataset> Size: 16GB
Dimensions:    (time: 394464, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_10U    (time, latitude, longitude) float32 16GB dask.array<chunksize=(33052, 82, 121), meta=np.ndarray>

In [9]:
V

<xarray.Dataset> Size: 16GB
Dimensions:    (time: 394464, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_10V    (time, latitude, longitude) float32 16GB dask.array<chunksize=(33052, 82, 121), meta=np.ndarray>

# flux

In [13]:
def q_flux(q, u, v):
    qu = q * u
    qv = q * v
    ds = xr.merge({'Q_U': qu, 'Q_V': qv})
    return ds

In [ ]:
qf_ds = q_flux(Q, U, V)
qf_ds

# divergence

In [14]:
def xr_div(u_da, v_ds):
    da_du = u_da.differentiate(coord='longitude')
    da_dv = v_ds.differentiate(coord='latitude')
    div = da_du + da_dv
    return div.rename('field_divergence')

In [ ]:
MFD = xr.div(qf_ds['Q_U'], qf_ds['Q_V'])
MFD

In [ ]:
MFD.to_netcdf('/glade/work/acruz/Caribbean_Heat_data/ERA5/MFD.nc')

In [21]:
client.shutdown()